In [ ]:
from pathlib import Path
import os, time, shutil, datetime

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from importlib import reload

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "core").exists() and (REPO_ROOT.parent / "core").exists():
    os.chdir(REPO_ROOT.parent)
    REPO_ROOT = Path.cwd()

from core import config, data, infer, net, pipelines, train
from helpers import utils

os.makedirs('figs', exist_ok=True)
os.makedirs('ckpts', exist_ok=True)

config.Config.configure_AMIDE_clinical_whole_brain()    

In [ ]:
# Internal use (full 3D brain volumes) - iterate for leave-1-out eval

# Papermill parameters: (make sure cell is tagged as parameters)
TRAINSET = 'vol7-9-10'  
TESTSET = 'vol8'        

do_train = False        # set False to skip training and used existing network
do_all_points = False   # set False to reduce evaluation to few min run (for MT)
do_amide = True         # set True to train and evaluate amide network (requires CEST data)

In [ ]:
# set False for internal use with complete brains dataset,
# otherwise use True for a one-slice-of-each-brain public sample
do_sample = True   
# !!! |--> public

xarr_name = './data/xarr_sample.nc' if do_sample else f'./data/{TRAINSET}.nc'
sfx = 'sample' if do_sample else TRAINSET

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_h%H-m%M")
# Transfer - use checkpoint trained on leave-one-out to test on a held-out from (e.g. 7,9,10 to 8)
ckpts = os.path.abspath(f'./ckpts/healthy/{sfx}_{timestamp}')  # edit for custom flows..
test_vol_name = 'xarr_sample' if do_sample else TESTSET

if do_sample and not do_train:
    ckpts = os.path.abspath(f'./ckpts/healthy/vol7-9-10_2026-04-28_h22-m45')  # edit for custom flows..
    

### Creating setup & inspecting the data

In [ ]:
data_xa = xr.open_dataset(xarr_name)
data_feed_mt_train = data.SlicesFeed.from_xarray(data_xa, seq_name='mt')
data_feed_amide_train = data.SlicesFeed.from_xarray(data_xa, seq_name='amide')

In [ ]:
reload(utils)

utils.viz_t1t2b0b1_sample_slices(data_xa, data_feed_mt_train, sample=do_sample)

## Training the PdVAE

Trainable neural encoder
(measured signal =>> posterior over quantitative params, specifically a gaussian around the mu which is the point estimate),

 with the decoder fixed as the (differentiable) Bloch-McConnell solver & protocol simulator

In [ ]:
import jax
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")

mt_sim_mode = 'isar2_c'  
data.SlicesFeed.norm_type = 'l2'

In [ ]:
results_folder = f'./figs/humans/{test_vol_name}_w_net_trained_on_{sfx}_{timestamp}/'
os.makedirs(results_folder, exist_ok=True)

print(results_folder)

In [ ]:
ckpts

In [ ]:
mt_train_folder = f'{ckpts}/mt'
cest_train_folder = f'{ckpts}/cest'
    
if do_train:
    t0 = time.time()
    shutil.rmtree(mt_train_folder, ignore_errors=True)
    if do_amide:
        shutil.rmtree(cest_train_folder, ignore_errors=True)
    predictor_mt, predictor_amide = pipelines.run_train(
        datafeed_mt=data_feed_mt_train, 
        datafeed_cest=data_feed_amide_train, 
        ckpt_folder=ckpts,     
        do_cest=do_amide,
        mt_sim_mode = mt_sim_mode
    )
    train_time = time.time() - t0 
else:
    predictor_mt = net.load_ckpt(folder=mt_train_folder)[0]
    if do_amide:
        predictor_amide = net.load_ckpt(folder=cest_train_folder)[0]
    train_time = np.nan


In [ ]:
print(f'Training time: {train_time:.2f} seconds')

In [ ]:
if do_train and do_amide:
    mt_trend = np.load(ckpts+'/mt_loss_trend.npy', allow_pickle=True).item()
    cest_trend = np.load(ckpts+'/cest_loss_trend.npy', allow_pickle=True).item()
    plt.figure(figsize=(10, 3))
    plt.loglog(np.arange(len( mt_trend['est_loss_history']))/13.5, mt_trend['est_loss_history'])
    plt.loglog(np.arange(len( cest_trend['est_loss_history']))/135/4, cest_trend['est_loss_history'])
    plt.xlim(10, 200)
    plt.figure(figsize=(10, 3))
    plt.semilogx(np.arange(len( mt_trend['est_loss_history'])), mt_trend['est_loss_history'])
    plt.semilogx(np.arange(len( cest_trend['est_loss_history'])), cest_trend['est_loss_history'])
    plt.ylim(3, 8)
    plt.xlim(100, 1e4)    
    
create_error_maps = False
if create_error_maps:  # the NRMSE heatmaps in Fig.7
    for vol in [7,9,10,8]:
        data_xa = xr.open_dataset(f'./data/vol{vol}.nc')
        data_feed_mt = data.SlicesFeed.from_xarray(data_xa, seq_name='mt')
        mask_gray, mask_white = data_xa.gray_mask.values, data_xa.white_mask.values
        data_feed_mt.slw=10
        transfer_res, err_mt, err_amide = pipelines.transfer_and_plot(
            brain2test_mt=data_feed_mt, brain2test_cest=None, #data_feed_amide, 
            predictor_mt=predictor_mt, predictor_cest=None, 
            figsfolder=f'./figs/humans/vol{vol}_w_net_trained_on_{sfx}', figsfx='mt', do_boxplots=False, slices=[20], 
            mask_gray=mask_gray, mask_white=mask_white,
            mt_simulation_mode=mt_sim_mode
        )
        folder = f'./figs/humans/vol{vol}_w_net_trained_on_{sfx}/'
        os.makedirs(folder, exist_ok=True)
        utils.draw_err_map(err_mt, err_amide, sli=-30, do_amide=False, folder=folder)


### Analyzing the resultant maps of tissue parameter estimates

In [ ]:
os.makedirs('./figs/humans', exist_ok=True)

In [ ]:
data_xa_test = xr.open_dataset(f'./data/{test_vol_name}.nc') 

data_feed_mt = data.SlicesFeed.from_xarray(data_xa_test, seq_name='mt')
data_feed_amide = data.SlicesFeed.from_xarray(data_xa_test, seq_name='amide')

mask_gray, mask_white = data_xa_test.gray_mask.values, data_xa_test.white_mask.values
    
data_feed_mt.slw=5

transfer_res, err_mt, err_amide = pipelines.transfer_and_plot(
    brain2test_mt=data_feed_mt, brain2test_cest=data_feed_amide, 
    predictor_mt=predictor_mt, predictor_cest=predictor_amide if do_amide else None, 
    figsfolder=f'{results_folder}', figsfx='mt', slices=list(range(4)) if do_sample else None,
    mask_gray=mask_gray, mask_white=mask_white, do_boxplots=False,
    mt_simulation_mode=mt_sim_mode
)

In [ ]:
create_white_gray_clouds = False
if create_white_gray_clouds:
    utils.create_white_gray_clouds()

In [ ]:
import analyze_uncertainty as au 
reload(au)

### Simplistic confidence maps using NRMSE

In [ ]:
reload(utils)
sli = 1 if do_sample else -30

utils.draw_err_map(err_mt, err_amide, sli=sli, do_amide=do_amide, folder=f'{results_folder}')

In [ ]:
mt_tissue_params_est, amide_tissue_param_est = transfer_res[1], transfer_res[4]
nn_pred_sim_signal_mt = transfer_res[2]

os.makedirs(f'{results_folder}/intermediates', exist_ok=True)

np.savez(
    f'{results_folder}/intermediates/mt_tissue_params_est.npz',
    mt_nn_pred_signal_normed_np=nn_pred_sim_signal_mt,    
    train_time=train_time,
    **mt_tissue_params_est,    
    )

amide_tissue_params_est = transfer_res[4] 
nn_pred_sim_signal_amide = transfer_res[5]
if do_amide:
    np.savez(
        f'{results_folder}/intermediates/amide_tissue_params_est.npz',
        amide_nn_pred_signal_normed_np=nn_pred_sim_signal_amide,    
        train_time=train_time,
        **amide_tissue_params_est,        
        )
    

## MT - posteriors by PS-VAE vs. likelihood mapping




### Visualize univariate uncertainty maps

In [ ]:
import matplotlib.patches as patches
import analyze_uncertainty as au; reload(au)

plt.rcParams.update({'font.size': 12})

fss_map = 100*mt_tissue_params_est['fc_T']
kss_map = mt_tissue_params_est['kc_T']

f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(mt_tissue_params_est, data_feed_mt.shape, is_cest=False)
    
axes = au.plot_CI_maps(
    f_est[:,sli,:][17:-8, 10:-5], f_sigma[:,sli,:][17:-8, 10:-5],
    k_est[:,sli,:][17:-8, 10:-5], k_sigma[:,sli,:][17:-8, 10:-5], is_cest=False, vmax_k=100, figsize=(9, 5)
)


points_white = [[77, 23], [77, 63], [30, 30], [30, 55]]
points_gray = [[74, 15], [74, 72], [43, 23], [38, 62]]
points = points_white + points_gray
labels = [0]*len(points_white) + [1]*len(points_gray)
points2disp = [(x-17, y-10) for (x,y) in points[4:2:-1]]   # !!!!
for jj, (label, point) in enumerate(zip(labels[4:2:-1], points2disp)):
    for ax in axes[:,1]:
        square = patches.Rectangle((point[1], point[0]), 4, 4, facecolor='none', edgecolor=['blue', 'gold'][label], linewidth=2)
        ax.add_patch(square)    
        ax.text(point[1]+1.5, point[0]-1.5, ['A','B'][jj], color=['gold', 'blue'][jj], fontsize=12, fontweight='bold', ha='center')

plt.savefig(f'{results_folder}/UQ_MT_CImaps.png', dpi=300, bbox_inches='tight')
print(points[4:2:-1])

### Visualize the point (per-voxel) multi-variate posteriors of the quantitative estimates

In [ ]:


_ = au.compare_posteriors(
    [(x,x) for x in range(20, 70, 5)], data_feed_mt, mt_tissue_params_est,
    mt_sim_mode=mt_sim_mode, sli=0, voxels2sample=8, folder_name=results_folder+'/MT_voxels', visualize=True
)

In [ ]:
# (!!) T1 map to manually choose points
if True:  
    plt.imshow(np.clip(data_xa.T1ms.data[:,sli,:]*data_feed_amide.roi_mask_nans[:,sli,:], 800, 2500))
    plt.minorticks_on()
    plt.grid('minor', which='both', color='gray', alpha=0.5, linestyle='--')

### Quantify PhiVAE vs. likelihood-mapping discrepancy

#### Residuals analysis while at it (might be a direction for future work..)

In [ ]:
## Residuals research - if looking for further problem-specific insight I'd start here..

os.makedirs(f'{results_folder}/mt_voxel', exist_ok=True)

fss_map = 100*mt_tissue_params_est['fc_T']
kss_map = mt_tissue_params_est['kc_T']

f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(mt_tissue_params_est, data_feed_mt.shape, is_cest=False)
    
jj = 0
for [_x, _y] in points:
    jj += 1
    f_best_dotprod, k_best_dotprod, dict_signal, nrmse, _df, _dk = au.get_nrmse_grid(
        None, _x, _y, sli, 
        data_feed_mt, mt_tissue_params_est,
        data_feed_cest=None, seq_name='mt',
        mt_sim_mode=mt_sim_mode
    )            
    try:        
        print(f'np.min(nrmse): {100*np.min(nrmse):.2f}%')  
        print(f'nrmse[NN-est]: {100*nrmse[int(f_est[_x, sli, _y]/100/_df), int(k_est[_x, sli, _y]/_dk)]:.2f}%')  
    except Exception as e:
        print(e)
    maha1, maha2, posterior_cov, CR_area, CIk_x_CIf,_ = au.viz_posteriors(
        f_est[_x, sli, _y], k_est[_x, sli, _y], 
        cov_nnpred_scaled[_x, sli, _y], 
        width[_x, sli, _y],  
        height[_x, sli, _y],  
        angle[_x, sli, _y],
        f_best_dotprod, k_best_dotprod, nrmse, _df, _dk,        
        is_cest=False, fontsize=14, 
        do_marginals=True, figsize=[7, 5], show_text=True, show_NN=True
        )    
    plt.savefig(f'{results_folder}/mt_voxel/{_x}_{sli}_{_y}.svg', bbox_inches='tight', dpi=300)
    plt.savefig(f'{results_folder}/mt_voxel/{_x}_{sli}_{_y}.png', bbox_inches='tight', dpi=300)
    
    plt.figure(figsize=(7,2))
    plt.plot(data_feed_mt.measured_normed_T[:, _x, sli, _y], 'o:')
    plt.plot(dict_signal[:, int(f_best_dotprod/100/_df), int(k_best_dotprod/_dk) ], 'o:')
    plt.title(f"{int(f_best_dotprod/100/_df)}, {int(k_best_dotprod/_dk)}")
    plt.grid()
    plt.figure(figsize=(7,2))
    plt.plot(data_feed_mt.measured_normed_T[:, _x, sli, _y], 'o:')
    plt.plot(dict_signal[:, int(f_est[_x, sli, _y]/100/_df), int(k_est[_x, sli, _y]/_dk) ], 'o:')
    plt.title(f"{int(f_est[_x, sli, _y]/100/_df)}, {int(k_est[_x, sli, _y]/_dk)}")
    plt.figure(figsize=(7,2))

    dd0 = data_feed_mt.measured_normed_T[:, _x, sli, _y] - \
        dict_signal[:, int(f_best_dotprod/100/_df), int(k_best_dotprod/_dk)]
    plt.plot(100*dd0, 'o:')
    dd = data_feed_mt.measured_normed_T[:, _x, sli, _y] - \
        dict_signal[:, int(f_est[_x, sli, _y]/100/_df), int(k_est[_x, sli, _y]/_dk) ]
    plt.plot(100*dd, 'rd-', linewidth=0.5)
    plt.title('Residuals (%)')
    plt.legend([
        f'dotprod: nrmse@MAP = {100*np.linalg.norm(dd0):.1f}', 
        f'NN: nrmse@mu = {100*np.linalg.norm(dd):.1f}'], 
        fontsize=10)
    plt.grid()
    plt.ylim(-1.5, 1.5)
    plt.show()

#### Run over large amount of voxels, collect Mahalanobis and other statistics


In [ ]:
reload(au)

def meshgrid_xy_pairs(shape):
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    xy_pairs = np.stack([x, y], axis=-1).reshape(-1, 2)
    return xy_pairs.tolist()

xy_points = meshgrid_xy_pairs(data_feed_mt.shape[0::2])

os.makedirs(results_folder+'/intermediates', exist_ok=True)
npz_filename = results_folder+'/intermediates/MT_UQ_stats_allpoints.npz'

if not do_all_points:
    xy_points = xy_points[::100]
    npz_filename = results_folder+'/intermediates/MT_UQ_stats_subset_of_points.npz'   
print(len(xy_points))    

mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings,\
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics_dict = \
        au.compare_posteriors(
            xy_points, data_feed_mt, mt_tissue_params_est,
            mt_sim_mode=mt_sim_mode, sli=sli, voxels2sample=20000, folder_name=results_folder+'/MT_voxels', visualize=False
        )       

np.savez_compressed(
    npz_filename,
    mahas1=mahas1, mahas2=mahas2,
    CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
    good_xy_points=good_xy_points, 
    f_est=f_est, k_est=k_est, cov_nnpred_scaled=cov_nnpred_scaled,
    covs_of_exact_posterior=covs_of_exact_posterior, modes_of_exact_posterior=modes_of_exact_posterior, means_of_exact_posterior=means_of_exact_posterior,
    timings=timings, train_time=train_time,
    exact_posterior_min_nrmse_percs=exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs=nn_posterior_min_nrmse_percs,
    **extra_metrics_dict
)

In [ ]:
saved = np.load(npz_filename, allow_pickle=True)
list(saved.keys())

In [ ]:
from IPython.display import Markdown, display

print('Sampled voxels: ', len(good_xy_points)), 
print("Before averaging voxels: ")
for mm in (4.5, 4, 3.5, 3):
    print(f"Mahas above {mm}: " +
        f"{100*sum(np.array(mahas1)>mm)/len(np.array(mahas1)):.1f}%, "+
        f"{100*sum(np.array(mahas2)>mm)/len(np.array(mahas2)):.1f}%"
    )
mean_mahas1 = np.reshape(mahas1, (len(good_xy_points), 10)).mean(axis=1)
mean_mahas2 = np.reshape(mahas2, (len(good_xy_points), 10)).mean(axis=1)
print("After averaging voxels: ")
for mm in (4.5, 4, 3.5, 3):
    print(f"Mahas above {mm}: " +
        f"{100*sum(np.array(mean_mahas1)>mm)/len(np.array(mean_mahas1)):.1f}%, "+
        f"{100*sum(np.array(mean_mahas2)>mm)/len(np.array(mean_mahas2)):.1f}%"
    )
    
mean_t = np.mean(timings)
std_t = np.std(timings)
latex_inline = rf"${mean_t:.2f}\pm{std_t:.2f}$"
print(latex_inline)
display(Markdown(f"Mean ± std (inline LaTeX): {latex_inline}"))
print(rf"whole slice: ${len(good_xy_points)*mean_t:.1f}\pm{np.sqrt(len(good_xy_points))*std_t:.1f} sec$")
print(rf"whole slice: ${len(good_xy_points)*mean_t/60:.1f}\pm{np.sqrt(len(good_xy_points))*std_t/60:.2f} mins$")
print(rf"whole brain (50 slices): ${50*len(good_xy_points)*mean_t/60/60:.1f}\pm{np.sqrt(50*len(good_xy_points))*std_t/60/60:.2f} hours$")

In [ ]:
au.summarize_mahas(mahas1, mahas2, good_xy_points) 
plt.xlabel('Mahalanobis Distance')
for format in ['svg', 'png']:       
    plt.savefig(results_folder+f'/MT_mahas_hist.{format}', bbox_inches='tight', dpi=300, format=format)

In [ ]:
## See prior versions for lots of experimental content and analyses (consult Alex)


## AMIDE - posteriors by PS-VAE vs. likelihood mapping

### Visualize univariate uncertainty maps

In [ ]:
fs_map = 100*mt_tissue_params_est['fb_T']
ks_map = mt_tissue_params_est['kb_T']

f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(amide_tissue_params_est, data_feed_mt.shape, is_cest=True)
    
axes = au.plot_CI_maps(
    f_est[:,sli,:][17:-8, 10:-5], f_sigma[:,sli,:][17:-8, 10:-5],
    k_est[:,sli,:][17:-8, 10:-5], k_sigma[:,sli,:][17:-8, 10:-5], is_cest=True, vmax_k=500, figsize=(9, 5)
)

for jj, (label, point) in enumerate(zip(labels[4:2:-1], points2disp)):
    for ax in axes[:,1]:
        square = patches.Rectangle((point[1], point[0]), 4, 4, facecolor='none', edgecolor=['blue', 'gold'][label], linewidth=2)
        ax.add_patch(square)    
        ax.text(point[1]+1.5, point[0]-1.5, ['A','B'][jj], color=['gold', 'blue'][jj], fontsize=12, fontweight='bold', ha='center')

plt.savefig(f'{results_folder}/UQ_AMIDE_CImaps.png', dpi=300, bbox_inches='tight')

### Visualize the point (per-voxel) multi-variate posteriors

In [ ]:

f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(mt_tissue_params_est, data_feed_mt.shape, is_cest=True)

_ = au.compare_posteriors(
    [(x,x) for x in range(20, 70, 5)], data_feed_mt, mt_tissue_params_est,
    cest_tissue_params_est=amide_tissue_param_est, is_cest=True, data_feed_cest=data_feed_amide, seq_name='amide',
    mt_sim_mode=mt_sim_mode, sli=0, voxels2sample=8, folder_name=results_folder+'/AMIDE_voxels', visualize=True
)

In [ ]:
## See prior versions for lots of experimental content and analyses (consult Alex)